# Experimental absorption to refractive index
Convert measured absorption to $\beta(E)$, extend it with a local Henke/CXRO dataset, and calculate $\delta(E)$ for $n(E)=1-\delta(E)+i\beta(E)$.

In [ ]:
from pathlib import Path

import numpy as np
from library import kramers_kronig as kk

In [ ]:
absorption_data = np.load("data/absorption_spectrum.npz")
energy_ev = absorption_data["energy_ev"]
absorption = absorption_data["absorption"]

# Complete absorption_to_beta options.
input_kind = "optical_depth"  # absorption_coefficient [m^-1], optical_depth, or transmission.
thickness_m = 100e-9           # Required for optical_depth/transmission; ignored for coefficient input.

beta_experimental = kk.absorption_to_beta(
    energy_ev,
    absorption,
    input_kind=input_kind,
    thickness_m=thickness_m,
)

In [ ]:
# Complete local Henke/CXRO text-loader options.
henke_file = "data/henke_optical_constants.txt"  # No network access is performed.
energy_column = 0   # Zero-based photon-energy column.
delta_column = 1    # Zero-based delta column.
beta_column = 2     # Zero-based beta column.
delimiter = None    # None means whitespace-separated text.
skiprows = 0        # Number of header rows to skip.

henke_energy_ev, henke_delta, henke_beta = (
    kk.load_henke_refractive_index(
        henke_file,
        energy_column=energy_column,
        delta_column=delta_column,
        beta_column=beta_column,
        delimiter=delimiter,
        skiprows=skiprows,
    )
)

# Complete refractive_index_from_beta_with_reference options.
transition_width_ev = 2.0  # Blend measured beta to reference at boundaries; zero means direct replacement.
return_extended = True     # Also return the merged energy grid, beta, and beta correction.

(
    beta,
    delta,
    extended_energy_ev,
    extended_beta,
    beta_correction,
) = kk.refractive_index_from_beta_with_reference(
    measured_energy_ev=energy_ev,
    measured_beta=beta_experimental,
    reference_energy_ev=henke_energy_ev,
    reference_delta=henke_delta,
    reference_beta=henke_beta,
    transition_width_ev=transition_width_ev,
    return_extended=return_extended,
)

In [ ]:
output_file = Path("data/refractive_index_constraints.npz")
np.savez(
    output_file,
    energy_ev=energy_ev,
    beta=beta,
    delta=delta,
    extended_energy_ev=extended_energy_ev,
    extended_beta=extended_beta,
    beta_correction=beta_correction,
)
print(f"Saved {output_file}")